# 101 — Download, Predict, Save Tiles

Downloads tiles one at a time, predicts lupine probability, saves:
- **Probability tile** (float32) per tile → `prob_tiles/`
- **Binary tile** (uint8) for each threshold in `PIXEL_THRESHOLDS` → `binary_tiles_t{X}/`
- Deletes downloaded source tile after prediction

Run `102_mosaic.ipynb` after this to merge binary tiles into a regional map.

**Requires:** `config.py` and `helpers.py` in the same folder.

In [ ]:
# ── Load config and helpers ───────────────────────────────────────────────────
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from config import *
from helpers import *

import multiprocessing
try:
    import psutil
    print(f"Cores: {N_CORES}  |  RAM: {RAM_GB:.0f} GB")
except:
    print(f"Cores: {N_CORES}")
print(f"BASE_DIR: {BASE_DIR}")
print(f"Thresholds: {PIXEL_THRESHOLDS}")
check_required_files()

In [ ]:
# ── Load model ────────────────────────────────────────────────────────────────
rf = joblib.load(MODEL_PATH)
rf.n_jobs = -1
rf.verbose = 0

print(f"Model:    {MODEL_NAME}")
print(f"Trees:    {rf.n_estimators}")
print(f"Features: {rf.n_features_in_}")
print(f"Cores:    {N_CORES} (all used)")

USE_NIR_CONTRAST = rf.n_features_in_ > 7
print(f"Model features: {rf.n_features_in_}  |  USE_NIR_CONTRAST: {USE_NIR_CONTRAST}")

In [ ]:
# ── Build tile list ───────────────────────────────────────────────────────────
print("── Building tile list ──────────────────────────────────────")

grid = gpd.read_file(GRID_PATH)
print(f"  Grid cells total: {len(grid)}")

if MODE == "region":
    gdf_regions = gpd.read_file(REGION_PATH)
    region_row  = gdf_regions[gdf_regions[REGION_ID_COL] == REGION_ID]
    if len(region_row) == 0:
        raise ValueError(f"Region {REGION_ID} not found")
    region_geom = region_row.to_crs(grid.crs).geometry.values[0]
    work_grid   = grid[grid.intersects(region_geom)].copy()
    print(f"  Mode: region {REGION_ID}  |  Overlapping tiles: {len(work_grid)}")
else:
    work_grid  = grid.copy()
    region_row = None
    print(f"  Mode: all tiles  |  Total: {len(work_grid)}")

url_map = {row['path'].split('/')[-1]: row['path']
           for _, row in work_grid.iterrows()}
all_fnames = sorted(url_map.keys())
print(f"  Tiles to process: {len(all_fnames)}")

In [ ]:
# ── Download → predict → save prob + binary tiles ────────────────────────────
print("── Processing tiles ────────────────────────────────────────")

# Resume — check which tiles are already done (use t=first threshold as reference)
ref_thresh      = PIXEL_THRESHOLDS[0]
ref_binary_dir  = binary_tile_dir(ref_thresh)
existing_binary = set(os.listdir(ref_binary_dir))
remaining       = [
    f for f in all_fnames
    if f.replace(".tif", f"_binary_t{ref_thresh}.tif") not in existing_binary
]
already_done = len(all_fnames) - len(remaining)

print(f"  Total:     {len(all_fnames)}")
print(f"  Done:      {already_done}")
print(f"  Remaining: {len(remaining)}")
print(f"  Saving prob tiles + binary at thresholds: {PIXEL_THRESHOLDS}")

skipped = 0
t0      = time.time()

for i, fname in enumerate(remaining):
    url             = url_map[fname]
    local_path      = os.path.join(TMP_DIR, fname)
    remaining_count = len(remaining) - i - 1

    # Download
    if not os.path.exists(local_path):
        ok = safe_download(url, local_path)
        if not ok:
            print(f"  [!] Download failed — skipping {fname}")
            skipped += 1
            continue

    # Read
    try:
        with rasterio.open(local_path) as src:
            img     = src.read().astype(np.float32)
            profile = src.profile.copy()
    except Exception as e:
        print(f"  [!] Read failed — skipping {fname}: {e}")
        skipped += 1
        try: os.remove(local_path)
        except: pass
        continue

    # Predict
    prob = rf.predict_proba(engineer_features(img, use_nir_contrast=USE_NIR_CONTRAST))[:, 1].astype(np.float32)
    prob_mask = prob.reshape(img.shape[1], img.shape[2])
    del img, prob; gc.collect()

    # Save probability tile (float32)
    prob_path = os.path.join(PROB_TILE_DIR, fname)
    prob_profile = profile.copy()
    prob_profile.update(count=1, dtype="float32",
                        compress="lzw", tiled=True,
                        blockxsize=256, blockysize=256)
    with rasterio.open(prob_path, "w", **prob_profile) as dst:
        dst.write(prob_mask[np.newaxis, :, :])

    # Save binary tile for each threshold
    for thresh in PIXEL_THRESHOLDS:
        binary    = (prob_mask >= thresh).astype(np.uint8)
        bin_fname = fname.replace(".tif", f"_binary_t{thresh}.tif")
        bin_path  = os.path.join(binary_tile_dir(thresh), bin_fname)
        bin_profile = profile.copy()
        bin_profile.update(count=1, dtype="uint8",
                           compress="lzw", tiled=True,
                           blockxsize=256, blockysize=256)
        with rasterio.open(bin_path, "w", **bin_profile) as dst:
            dst.write(binary[np.newaxis, :, :])
        del binary

    del prob_mask; gc.collect()

    # Delete source tile
    if DELETE_AFTER_PREDICT:
        try: os.remove(local_path)
        except: pass

    # Progress report
    elapsed = time.time() - t0
    done    = i + 1 - skipped
    rate    = done / elapsed if elapsed > 0 else 0
    eta     = remaining_count / rate if rate > 0 else 0

    if (i + 1) % 10 == 0 or (i + 1) == len(remaining):
        remaining_time = eta  # seconds left at current rate
        print(f"\n  ── Checkpoint [{i+1}/{len(remaining)}] ──────────────────")
        print(f"     Elapsed:   {elapsed/3600:.1f}h")
        print(f"     Rate:      {rate*3600:.0f} tiles/h")
        print(f"     Remaining: {remaining_time/3600:.1f}h")
        for t in PIXEL_THRESHOLDS:
            n = len(os.listdir(binary_tile_dir(t)))
            print(f"     Saved t={t}: {n} tiles")
        print()

print(f"\n  ✓ Done. Skipped: {skipped}  |  Time: {(time.time()-t0)/3600:.1f}h")
print(f"  Prob tiles saved to: {PROB_TILE_DIR}")
for t in PIXEL_THRESHOLDS:
    n = len(os.listdir(binary_tile_dir(t)))
    print(f"  Binary tiles t={t}: {n} tiles → {binary_tile_dir(t)}")